In [1]:
import numpy as np
import pandas as pd

number = int | float

## Series

1-dim data structure (similar to numpy arrays). Major difference is in the access pattern: it allows access based on _lebels_ not just positions.

In [2]:
labels = ['l1', 'l2', 'l3']
data = [10, 20, 30]

You can create a series directly from raw data (pyhton lists or np arrays), and the index will be automatically generated (as integers).

In [3]:
series = pd.Series(data)
print(series)

# access is positional in this case just like for np arrays
series[0]

0    10
1    20
2    30
dtype: int64


np.int64(10)

But you can also explicitly set the labels (i.e. the so-called `Index`).

In [4]:
series = pd.Series(data=data, index=labels)
print(series)

# access is now label-based
series['l1']


l1    10
l2    20
l3    30
dtype: int64


np.int64(10)

You can create a labeles series directly from a Python dictionary.

In [5]:
data = { 'l1': 10, 'l2': 20, 'l3': 30}
series = pd.Series(data)

print(series)

l1    10
l2    20
l3    30
dtype: int64


You can hold any type of object as values in a series (even function objects).

In [6]:
series = pd.Series(data=[0, "one", print])
print(series)

0                            0
1                          one
2    <built-in function print>
dtype: object


Operations involving multiple series are "label aligned": where labels don't match il will put null (NaN in the example below).

In [7]:
s1 = pd.Series({"a": 1, "b": 2})
s2 = pd.Series({"a": 1, "c": 3})
s3 = s1 + s2

print(s3)

a    2.0
b    NaN
c    NaN
dtype: float64


The presence of `NaN` causes the `s3` series above to be upcast to `float64`; `int` series cannot hold `NaN`s, but `Int64` can (use `.astype` to perform type casting):

In [8]:
print(s3.astype('Int64'))

a       2
b    <NA>
c    <NA>
dtype: Int64


## DataFrame's

### FataFrame creation

A `DataFrame` is a 2D data structures with columns and rows (row labels). Think of it as an excel table. Technically, a DataFrame is a set of series (columns) sharing the same row labels (indexes).

In [9]:
df = pd.DataFrame(
    data=np.random.randn(5, 4), # 5 rows and 4 cols
    index=["A", "B", "C", "D", "E"], # row labels
    columns=["X", "Y", "Z", "W"] # col labels
)

display(df)

,X,Y,Z,W
A,-2.642644,0.087589,0.521543,1.064099
B,0.228586,0.284648,-1.695408,-0.535062
C,-0.558994,1.042518,-1.428729,1.619556
D,0.378218,0.186777,0.493080,1.278330
E,0.211304,-0.311615,-1.262426,0.101256


Creating data-frames out of dictionaries:

In [10]:
df = pd.DataFrame({
    "col_1": [10, 20],
    "col_2": ["A", "B"]
}, index=["row_1", "row_2"])

# same as
df = pd.DataFrame({
    "col_1": { "row_1": 10, "row_2": 20 },
    "col_2": { "row_1": "A", "row_2": "B" },
})

# same as (row-centric)
df = pd.DataFrame.from_dict({
    "row_1": [10, "A"],
    "row_2": [20, "B"]
}, columns=["col_1", "col_2"], orient="index")

display(df)

,col_1,col_2
row_1,10,A
row_2,20,B


You can _reset the index_ (switch the the default positional index) - while the previous index is transformed into an "index" column:

In [11]:
df = df.reset_index()

display(df)

,index,col_1,col_2
0,row_1,10,A
1,row_2,20,B


We can also _set the index_ to an existing column in the data-frame:

In [12]:
df = pd.DataFrame({
    "name": ["Andrei", "Radu", "George"],
    "age": [10, 20, 30]
})

df = df.set_index("name")

display(df)

,age
name,
Andrei,10
Radu,20
George,30


You can store arbitrary objects in data-frames as long as they are "hash-able". This pattern imposes some significant perf. penalties - but it can be really powerful.

An interesting pattern is to store functions (i.e. logic) as values in a data-frame (in a column with a type of `object`).

In [13]:
def double(x: number) -> number: return x * 2
def square(x: number) -> number: return x ** 2

df = pd.DataFrame({
    'data': [10, 20, 30],
    'logic': [double, square, double]
})

display(df)

,data,logic
0,10,<function double at 0x78eb00c54860>
1,20,<function square at 0x78eb00c547c0>
2,30,<function double at 0x78eb00c54860>


In [14]:
def row_processor(row: pd.Series) -> number: return row["logic"](row["data"])

df["processed_data"] = df.apply(row_processor, axis=1)

display(df)

,data,logic,processed_data
0,10,<function double at 0x78eb00c54860>,20
1,20,<function square at 0x78eb00c547c0>,400
2,30,<function double at 0x78eb00c54860>,60


### DataFrame selection

Selecting individual columns gives us the corresponding pandas series:

In [15]:
display(df["data"])

0    10
1    20
2    30
Name: data, dtype: int64

One can selects multiple cols at the same time (the result will be another data-frame) by passing a **list** of column names as the value of the index into the original data-frame: 

In [16]:
display(df[["data", "processed_data"]])

,data,processed_data
0,10,20
1,20,400
2,30,60


Selecting rows can be done by label or by position:

In [17]:
display(df.loc[0]) # this gives you back a series

display(df.loc[[0]]) # this gives you back a data-frame instead

data                                               10
logic             <function double at 0x78eb00c54860>
processed_data                                     20
Name: 0, dtype: object

,data,logic,processed_data
0,10,<function double at 0x78eb00c54860>,20


In [18]:
display(df.iloc[0])
display(df.iloc[[0, 1]])

data                                               10
logic             <function double at 0x78eb00c54860>
processed_data                                     20
Name: 0, dtype: object

,data,logic,processed_data
0,10,<function double at 0x78eb00c54860>,20
1,20,<function square at 0x78eb00c547c0>,400


### Adding and removing rows / cols

In [19]:
df = pd.DataFrame(
    data=np.random.rand(20).reshape(5,4),
    columns=["X", "Y", "Z", "W"],
    index=["A", "B", "C", "D", "E"]
)

display(df)

,X,Y,Z,W
A,0.163412,0.313371,0.509257,0.169452
B,0.442468,0.510412,0.287460,0.884876
C,0.255181,0.171881,0.248856,0.638036
D,0.674648,0.159129,0.177087,0.919767
E,0.354400,0.747557,0.131541,0.882794


Creating a new column:

In [20]:
df["T"] = df["X"] + df["Y"]

display(df)

,X,Y,Z,W,T
A,0.163412,0.313371,0.509257,0.169452,0.476782
B,0.442468,0.510412,0.287460,0.884876,0.952880
C,0.255181,0.171881,0.248856,0.638036,0.427062
D,0.674648,0.159129,0.177087,0.919767,0.833777
E,0.354400,0.747557,0.131541,0.882794,1.101957


You can delete columns (returns a **new** data-frame - unless you pass in `inplace=True`).

**NOTE**: using `inplace=True` is _strongly discouraged_ in recent versions of pandas - use explicit reassignment of the data-frame instead.

In [21]:
display(df.drop(columns="T"))

# same as:
# df.drop("T", axis=1)


,X,Y,Z,W
A,0.163412,0.313371,0.509257,0.169452
B,0.442468,0.510412,0.287460,0.884876
C,0.255181,0.171881,0.248856,0.638036
D,0.674648,0.159129,0.177087,0.919767
E,0.354400,0.747557,0.131541,0.882794


You can delete **rows** in a similar fashion:

In [22]:
display(df.drop("A"))

# same as
# df.drop("A", axis=0)

,X,Y,Z,W,T
B,0.442468,0.510412,0.287460,0.884876,0.952880
C,0.255181,0.171881,0.248856,0.638036,0.427062
D,0.674648,0.159129,0.177087,0.919767,0.833777
E,0.354400,0.747557,0.131541,0.882794,1.101957


Data-frames use numpy arrays under the hood:

In [23]:
df.shape

(5, 5)

### Filtering via conditional selection

You can **filter rows** by using conditional selection:

In [24]:
df = pd.DataFrame({
    "name": ["Andrei", "Radu", "George"],
    "age": [10, 20, 30]
})

# filter the adults
df = df[df["age"] > 18]

# apply multiple conditions (with & / |)
df = df[(df["age"] > 18) & (df["age"] < 25)]

display(df)

,name,age
1,Radu,20


### Multi-level indexes

In [25]:
products: list[tuple[str, str]] = [
    ("Sports", "Table tennis"),
    ("Sports", "Soccer ball"),
    ("Sports", "Baseball glove"),

    ("Clothing", "Men's shirt"),
    ("Clothing", "Shorts"),
    ("Clothing", "Socks"),

    ("Electronics", "TV"),
    ("Electronics", "PS5"),
    ("Electronics", "XBox"),
]

multi_index = pd.MultiIndex.from_tuples(products)

data = [
    [125.0, True],
    [18.99, True],
    [22.99, False],

    [25.0, True],
    [8.99, False],
    [2.99, False],

    [255.0, True],
    [299.99, True],
    [249.99, False],
]

df = pd.DataFrame(data, index=multi_index, columns=["price", "availability"])
df.index.names = ["category", "product"]

display(df)

price  availability
category    product                             
Sports      Table tennis    125.00          True
            Soccer ball      18.99          True
            Baseball glove   22.99         False
Clothing    Men's shirt      25.00          True
            Shorts            8.99         False
            Socks             2.99         False
Electronics TV              255.00          True
            PS5             299.99          True
            XBox            249.99         False

Selection operations in multi-leveled indexed data-frames:

In [26]:
result = df.loc["Clothing"].loc["Shorts"]

display(result)

price            8.99
availability    False
Name: Shorts, dtype: object

Using cross-sections:

In [27]:
result = df.xs("Socks", level="product")

display(result)

,price,availability
category,,
Clothing,2.99,False


In [28]:
df = df[df["availability"]]

display(df)

price  availability
category    product                           
Sports      Table tennis  125.00          True
            Soccer ball    18.99          True
Clothing    Men's shirt    25.00          True
Electronics TV            255.00          True
            PS5           299.99          True

### Data cleanup and sanitization

Handling missing data:

In [29]:
df = pd.DataFrame({
    "price": [100.0, 55.3, np.nan],
    "quantity": [3, np.nan, np.nan],
    "availability": [True, True, False]
})

df["quantity"] = df["quantity"].astype(dtype="Int32")

display(df)

,price,quantity,availability
0,100.0,3,True
1,55.3,<NA>,True
2,NaN,<NA>,False


In [30]:
# drop all the rows with at least one missing column value
display(df.dropna())


,price,quantity,availability
0,100.0,3,True


In [31]:
# drop all the rows with at least "threshold no" of missing column values
display(df.dropna(thresh=2))

,price,quantity,availability
0,100.0,3,True
1,55.3,<NA>,True


In [32]:
# provide a default value for missing column data
display(df.fillna(0.0))

# logic can be column-specific
display(df.fillna({"price": df["price"].mean(), "quantity": 0}))

,price,quantity,availability
0,100.0,3,True
1,55.3,0,True
2,0.0,0,False


,price,quantity,availability
0,100.00,3,True
1,55.30,0,True
2,77.65,0,False


Other data cleanup related ops:

In [33]:
# 1. drop_duplicates: Removing redundant rows
df_dupes = pd.DataFrame({"A": [1, 1, 2], "B": ["x", "x", "y"]})
display(df_dupes.drop_duplicates())

,A,B
0,1,x
2,2,y


In [34]:
# 2. replace: Global value substitution
display(df_dupes.replace("x", "Z"))

,A,B
0,1,Z
1,1,Z
2,2,y


In [35]:
# 3. to_numeric: Safe type conversion with error coercion
s = pd.Series(["1.0", "2.1", "invalid"])
display(pd.to_numeric(s, errors="coerce"))

0    1.0
1    2.1
2    NaN
dtype: float64

In [36]:
# 4. .str accessors: Vectorized string operations
names = pd.Series([" andrei ", "RADU", " George"])
display(names.str.strip().str.title())

0    Andrei
1      Radu
2    George
dtype: str

### Grouping and aggregation

In [37]:

products = [
    ["Sports", "Table tennis", 125.0, True],
    ["Sports", "Soccer ball", 18.99, True],
    ["Sports", "Baseball glove", 22.99, False],

    ["Clothing", "Men's shirt", 25.0, True],
    ["Clothing", "Shorts", 8.99, False],
    ["Clothing", "Socks", 2.99, False],

    ["Electronics", "TV", 255.0, True],
    ["Electronics", "PS5", 299.99, True],
    ["Electronics", "XBox", 249.99, False],
]

df = pd.DataFrame(
    data=products, 
    columns=["category", "name", "price", "availability"]
)

display(df)

,category,name,price,availability
0,Sports,Table tennis,125.00,True
1,Sports,Soccer ball,18.99,True
2,Sports,Baseball glove,22.99,False
3,Clothing,Men's shirt,25.00,True
4,Clothing,Shorts,8.99,False
5,Clothing,Socks,2.99,False
6,Electronics,TV,255.00,True
7,Electronics,PS5,299.99,True
8,Electronics,XBox,249.99,False


When grouping and aggregating (e.g., calling `.mean()`), modern Pandas requires `numeric_only=True` if the DataFrame contains non-numeric columns like strings. Otherwise, it will raise a `TypeError`.

In [38]:
# Group by category and calculate mean price
# We MUST use numeric_only=True to ignore the 'name' and 'availability' columns
avg_prices = df.groupby("category").mean(numeric_only=True)

display(avg_prices)

,price,availability
category,,
Clothing,12.326667,0.333333
Electronics,268.326667,0.666667
Sports,55.660000,0.666667


### Data-frame concatenation and merging

#### Concatenation

Concatenation is done via a call to `pd.concat()`; you can control the axis of the concatenation, but you should pay attention to the shape of the data-frames being stiched together.

Row-wise concatenation:

In [39]:
left_df = pd.DataFrame(data=np.arange(1, 10).reshape(3,3), columns=["A", "B", "C"])
right_df = pd.DataFrame(data=np.arange(11, 20).reshape(3,3), columns=["A", "B", "C"])

result = pd.concat([left_df, right_df])
display(result)

,A,B,C
0,1,2,3
1,4,5,6
2,7,8,9
0,11,12,13
1,14,15,16
2,17,18,19


Column-wise concatenation:

In [40]:
left_df = pd.DataFrame(data=np.arange(1, 10).reshape(3,3), columns=["A", "B", "C"])
right_df = pd.DataFrame(data=np.arange(11, 20).reshape(3,3), columns=["D", "E", "F"])

result = pd.concat([left_df, right_df], axis=1)

display(result)

,A,B,C,D,E,F
0,1,2,3,11,12,13
1,4,5,6,14,15,16
2,7,8,9,17,18,19


#### Merging and joining

Merging is very similar with SQL table joins; you need to provide the key to join on (via the `on=` keyword param). You can also control the type of join (the regular inner/outer/left/right apply - default is `inner`).

In [41]:
left_df = pd.DataFrame(data=np.arange(1, 10).reshape(3,3), columns=["A", "B", "C"])
left_df["key"] = ["k0", "k1", "k2"]

right_df = pd.DataFrame(data=np.arange(11, 20).reshape(3,3), columns=["D", "E", "F"])
right_df["key"] = ["k0", "k1", "k2"]

result = left_df.merge(right_df, how="inner", on=["key"])

display(result)

,A,B,C,key,D,E,F
0,1,2,3,k0,11,12,13
1,4,5,6,k1,14,15,16
2,7,8,9,k2,17,18,19


The merge puts the key column in the middle. To reorder columns, simply reassign the DataFrame with a list of column names in the desired order:

In [42]:
# Move "key" to the front
reordered = result[["key", "A", "B", "C", "D", "E", "F"]]

# Or more generally:
# df = df[["key"] + [col for col in df.columns if col != "key"]]

display(reordered)

,key,A,B,C,D,E,F
0,k0,1,2,3,11,12,13
1,k1,4,5,6,14,15,16
2,k2,7,8,9,17,18,19


In [43]:
reordered = result[["key"] + [col for col in result.columns if col != "key"]]

display(reordered)

,key,A,B,C,D,E,F
0,k0,1,2,3,11,12,13
1,k1,4,5,6,14,15,16
2,k2,7,8,9,17,18,19


If the key you are joining on is the actual `index`, there's a shotened version of `merge()` called `join()`:

In [44]:
left_df = pd.DataFrame(
    data=np.arange(1, 10).reshape(3, 3),
    columns=["A", "B", "C"],
    index=["k0", "k1", "k2"],
)

right_df = pd.DataFrame(
    data=np.arange(11, 20).reshape(3, 3),
    columns=["D", "E", "F"],
    index=["k0", "k1", "k2"],
)

result = left_df.join(right_df, how="inner")

display(result)

,A,B,C,D,E,F
k0,1,2,3,11,12,13
k1,4,5,6,14,15,16
k2,7,8,9,17,18,19


### Misc data-frame operations

In [45]:
df_1 = pd.DataFrame(
    data=np.arange(1, 26).reshape(5, 5),
    columns=["A", "B", "C", "D", "E"]
)

df_2 = pd.DataFrame(
    data=np.arange(21, 46).reshape(5, 5),
    columns=["A", "B", "C", "D", "E"]
)

df = pd.concat([df_1, df_2]).reset_index(drop=True)

display(df)

,A,B,C,D,E
0,1,2,3,4,5
1,6,7,8,9,10
2,11,12,13,14,15
3,16,17,18,19,20
4,21,22,23,24,25
5,21,22,23,24,25
6,26,27,28,29,30
7,31,32,33,34,35
8,36,37,38,39,40
9,41,42,43,44,45


Get the list of columns in a data-frame:

In [46]:
df.columns

Index(['A', 'B', 'C', 'D', 'E'], dtype='str')

Finding unique values in a column:

In [47]:
# find unique values in column
display(df["A"].unique())

# how many unique values there are in acolumn
display(df["A"].nunique())

array([ 1,  6, 11, 16, 21, 26, 31, 36, 41])

9

In [48]:
# compute value frequency in a column
display(df["A"].value_counts().reset_index())

,A,count
0,21,2
1,1,1
2,6,1
3,11,1
4,16,1
5,26,1
6,31,1
7,36,1
8,41,1


Create a new column based on the value of another column; use `map()` mainly to work on `Series` and `apply()` to work on `DataFrame` (don't forget `axis=1` param to be able to work at the row level).

**NOTE**: `apply()` is very versatile and quite useful (you'll be using it a lot); very similar with user-defined-functions (UDFs) in Spark.

In [49]:
def process_row(row: pd.Series) -> number:
    return sum([row[letter] for letter in "A B C D E".split()])

df["as_str"] = df["E"].astype("str")
df["len"] = df["as_str"].map(len)
df["sum"] = df.apply(process_row, axis=1)

display(df)

result = df[df["as_str"].str.startswith("1")]

display(result)

,A,B,C,D,E,as_str,len,sum
0,1,2,3,4,5,5,1,15
1,6,7,8,9,10,10,2,40
2,11,12,13,14,15,15,2,65
3,16,17,18,19,20,20,2,90
4,21,22,23,24,25,25,2,115
5,21,22,23,24,25,25,2,115
6,26,27,28,29,30,30,2,140
7,31,32,33,34,35,35,2,165
8,36,37,38,39,40,40,2,190
9,41,42,43,44,45,45,2,215


,A,B,C,D,E,as_str,len,sum
1,6,7,8,9,10,10,2,40
2,11,12,13,14,15,15,2,65


Sorting:

In [50]:
result = df.sort_values(by="as_str", ascending=False)

display(result)

,A,B,C,D,E,as_str,len,sum
0,1,2,3,4,5,5,1,15
9,41,42,43,44,45,45,2,215
8,36,37,38,39,40,40,2,190
7,31,32,33,34,35,35,2,165
6,26,27,28,29,30,30,2,140
4,21,22,23,24,25,25,2,115
5,21,22,23,24,25,25,2,115
3,16,17,18,19,20,20,2,90
2,11,12,13,14,15,15,2,65
1,6,7,8,9,10,10,2,40


Finding null values:

In [51]:
display(df.isnull()) # returns a "boolean mask" (for conditional selection)

,A,B,C,D,E,as_str,len,sum
0,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False
5,False,False,False,False,False,False,False,False
6,False,False,False,False,False,False,False,False
7,False,False,False,False,False,False,False,False
8,False,False,False,False,False,False,False,False
9,False,False,False,False,False,False,False,False


Pivot tables:

In [52]:
data = {
    "A": ["foo", "foo", "foo", "bar", "bar", "bar"],
    "B": ["one", "one", "two", "two", "one", "one"],
    "C": ["x", "y", "x", "y", "x", "y"],
    "D": [1, 3, 2, 5, 4, 1],
}

df = pd.DataFrame(data)

display(df)

,A,B,C,D
0,foo,one,x,1
1,foo,one,y,3
2,foo,two,x,2
3,bar,two,y,5
4,bar,one,x,4
5,bar,one,y,1


In [53]:
piv_table = pd.pivot_table(df, index=["A", "B"], columns="C", values="D")

display(piv_table)

C          x    y
A   B            
bar one  4.0  1.0
    two  NaN  5.0
foo one  1.0  3.0
    two  2.0  NaN

## Loading / Saving data

Data can be loaded / saved from / to various formats. Here we will only work with the CSV format.

### Loading data

Pandas provides a set of `pd.read_*` methods you can use to load data from various sources:

In [54]:
df = pd.read_csv("./data/input/people.csv")

display(df)

,name,gender,age
0,Radu,M,22
1,Mihai,M,32
2,Jane,F,56
3,Jill,F,12
4,Geroge,M,41


In [55]:
df["gender_cat"] = df["gender"].map({"M": "male", "F": "female"})
df["gender_cat"] = df["gender_cat"].astype("category")
display(df)

,name,gender,age,gender_cat
0,Radu,M,22,male
1,Mihai,M,32,male
2,Jane,F,56,female
3,Jill,F,12,female
4,Geroge,M,41,male


### Saving data

Pandas provides a set of `df.to_*` methods you can use to save the data in various formats:

In [56]:
# pass in index=False, to avoid including the index column in the output
df.to_csv("./data/output/people.csv", index=False)